In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### Affinity translator

In [ ]:
import numpy as np

# Simulating real fMRI data
# 50 brain regions, monitored over 200 time points
# Some regions will intentionally "flash together"
np.random.seed(42)

# Generates random noise
brain_signals = np.random.randn(50, 200)

# Creates an artificial "Functional Network":
# We force Regions 0, 1, 2 and 3 to be identical (Strong Cluster)
brain_signals[1] = brain_signals[0] + np.random.normal(0, 0.1, 200)
brain_signals[2] = brain_signals[0] + np.random.normal(0, 0.1, 200)

In [ ]:
from interface.modeling_logic.translators import AffinityTranslator

# Initializes the Translator
# threshold=0.5: Ignores weak correlations
# regularization=0.5: Charges a fee of 0.5 energy to connect a node
translator = AffinityTranslator(threshold=0.5, regularization=0.5)

# Translate to Matrix Q
print("Translating brain signals to QUBO...")
q_matrix_rust = translator.from_time_series(brain_signals)

# Visualizing the first data to check
print("\n--- Excerpt from Matrix Q (First 4x4) ---")
for row in q_matrix_rust[:4]:
    print([f"{x:.2f}" for x in row[:4]])

print(f"\nMatrix ready to send to solver! Size: {len(q_matrix_rust)}x{len(q_matrix_rust)}")

In [ ]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(q_matrix_rust))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(q_matrix_rust))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

In [ ]:
import core_qyra_solver
sol = core_qyra_solver.solve(q_matrix_rust, priors_list, 1000)
print(sol)

### Conflict translator

In [6]:
from interface.modeling_logic.translators import ConflictTranslator
# Scenario: You have 5 possible meetings to attend today.
# But some of them happen at the same time (Conflict).

meetings_num = 5

# Weights: Some meetings are more important (Priority)
# Meeting 0 is worth a lot (3.0), the others are worth the normal amount (1.0)
importance = [3.0, 1.0, 1.0, 1.0, 1.0]

# List of Conflicts (Which meetings overlap in time)
# (0, 1): Meeting 0 and 1 are at the same time.
# (0, 2): Meeting 0 and 2 are at the same time.
# (3, 4): Meeting 3 and 4 are at the same time.
conflicts = [(0, 1), (0, 2), (3, 4)]

# Initialize the translator with a High Penalty
# Since meeting 0 is worth 3.0 and meeting 1 is worth 1.0, the sum is 4.0.
# The penalty must be GREATER than 4.0, otherwise it will choose both and "profit" from the balance.
translator = ConflictTranslator(penalty=5.0)

q_matrix = translator.from_conflict_list(meetings_num, conflicts, weights=importance)

print("--- Conflict Matrix (Independent Set) ---")
print("      R0     R1     R2     R3     R4")
for i, row in enumerate(q_matrix):
    row_str = "  ".join([f"{x:5.1f}" for x in row])
    print(f"R{i}: {row_str}")

print("\nInterpretation:")
print("- Negative values ​​on the diagonal: 'Please choose me!'")
print("- Values ​​of 5.0 outside the diagonal: 'NEVER choose us two together!'")

--- Conflict Matrix (Independent Set) ---
      R0     R1     R2     R3     R4
R0:  -3.0    5.0    5.0    0.0    0.0
R1:   5.0   -1.0    0.0    0.0    0.0
R2:   5.0    0.0   -1.0    0.0    0.0
R3:   0.0    0.0    0.0   -1.0    5.0
R4:   0.0    0.0    0.0    5.0   -1.0

Interpretation:
- Negative values ​​on the diagonal: 'Please choose me!'
- Values ​​of 5.0 outside the diagonal: 'NEVER choose us two together!'


In [7]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(q_matrix))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(q_matrix))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

Running GNN to predict solution structure...


In [8]:
import core_qyra_solver
sol = core_qyra_solver.solve(q_matrix, priors_list, 1000)
print(sol)

(b'\x01\x00\x00\x01\x00', -4.0)


Optimizing...
Optimization completed.


### One-Hot Translator

In [6]:
from interface.modeling_logic.translators import OneHotTranslator
import numpy as np


def build_tsp_matrix(distance_matrix, penalty=10.0):
    """
    Construct the complete QUBO matrix for a TSP.
    Variables = N Cities * N Time Steps.
    """
    num_cities = len(distance_matrix)
    num_vars = num_cities * num_cities # Grid N x N
    
    # Initializes empty matrix
    q_matrix = np.zeros((num_vars, num_vars))
    translator = OneHotTranslator(penalty=penalty)

    print(f"Creating TSP with {num_cities} cities ({num_vars} variables)...")

    # --- PART 1: The Rules of the Game (Constraints) ---

    # Rule A: At each point in time (step), I must be in ONE city.
    # (Constraint on the columns of the imaginary grid)
    for t in range(num_cities):
        # Indices of the variables for time 't': [0, 3, 6] for 3x3
        time_indices = [t * num_cities + c for c in range(num_cities)]
        translator.apply_one_hot(q_matrix, time_indices)

    # Rule B: Each city must be visited EXACTLY ONCE. 
    # (Constraint on the lines of the imaginary grid)
    for c in range(num_cities):
        # Indices of city 'c' at all times: [0, 1, 2] for city 0
        city_indices = [t * num_cities + c for t in range(num_cities)]
        translator.apply_one_hot(q_matrix, city_indices)

    # --- PART 2: The Objective (Minimizing Distance) ---
    # We add the distance costs to the Q matrix.
    # If x_{t,i} and x_{t+1,j} are 1, we add the distance(i, j).
    
    for t in range(num_cities - 1): # For each time step (except the last one)
        for i in range(num_cities): # Origin
            for j in range(num_cities): # Destination
                if i == j: continue 

                # Identifies the indices in the linear vector
                u = t * num_cities + i       # I'm in the city and time t
                v = (t + 1) * num_cities + j # I am in the city at time t+1.
                
                dist = distance_matrix[i][j]
                
                # Add a small cost to encourage shorter routes
                # Important: The cost must be less than the Penalty!
                q_matrix[u][v] += dist * 0.5 
                q_matrix[v][u] += dist * 0.5

    return q_matrix.tolist()

# --- SIMULATION ---
# 3 Cities (A, B, C)
# Distance matrix (symmetric)
# A-B = 10, A-C = 20, B-C = 5
distances = [
    [0, 10, 20],
    [10, 0, 5],
    [20, 5, 0]
]

# Penalty 50.0 (much larger than the greatest distance, 20)
tsp_q_matrix = build_tsp_matrix(distances, penalty=50.0)

print("\nGenerated matrix Q (First 5 rows):")
for row in tsp_q_matrix[:5]:
    print(["{:.1f}".format(x) for x in row[:5]])
    
print(f"\nTotal Size: {len(tsp_q_matrix)}x{len(tsp_q_matrix)}")

Creating TSP with 3 cities (9 variables)...

Generated matrix Q (First 5 rows):
['-100.0', '50.0', '50.0', '50.0', '5.0']
['50.0', '-100.0', '50.0', '5.0', '50.0']
['50.0', '50.0', '-100.0', '10.0', '2.5']
['50.0', '5.0', '10.0', '-100.0', '50.0']
['5.0', '50.0', '2.5', '50.0', '-100.0']

Total Size: 9x9


In [7]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(tsp_q_matrix))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(tsp_q_matrix))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

Running GNN to predict solution structure...


In [8]:
import core_qyra_solver
sol = core_qyra_solver.solve(tsp_q_matrix, priors_list, 1000)
print(sol)

(b'\x00\x00\x01\x00\x01\x00\x01\x00\x00', -285.0)


Optimizing...
Optimization completed.
